# 构建能记住用户的智能体

大多数智能体每次对话都从零开始。顾客告诉你的购物助手自己的尺码、预算以及他们回避哪些材质，可下次他们再来时，智能体已经把一切忘光了。他们不得不重复自己说过的话，体验显得千篇一律而缺乏个性化。

我们刚刚在 Claude Managed Agents 中推出了记忆（Memory）功能来解决这个问题。可以把它理解为智能体为每位客户配备的一本共享笔记本：Claude 在会话期间记下相关内容，等同一位客户下次再来时，那些笔记仍然在那里。配置只需一次简单的 API 调用。

在本指南中，你将为一个零售品牌构建一个示例购物助手。该智能体会在顾客首次光临时了解其偏好，把它们保存到对应该用户的记忆存储中，并在下次光临时自动调取，无需顾客再告知一次。

## 你将构建什么

- 一个保存某位客户购物偏好的**记忆存储**
- 一个被配置为查看并更新该存储的**购物智能体**
- **两个独立的会话**，用来演示记忆如何跨次访问延续
- 一种从你自己的应用中**查看与预置**记忆的模式

## 记忆是如何工作的

当你把一个记忆存储挂到会话上时，它会以目录的形式出现在智能体环境中的 `/mnt/memory/{store-name}` 路径下，Claude 用它的标准文件工具在那里读写文件。你的应用通过 REST API 对同一批文件拥有完整的读写权限，因此你可以用已知事实预置一个存储、审计智能体写下的内容，或把全部内容导出到你自己的系统。

> **Beta 功能。** 记忆存储是 Claude Managed Agents 公开 beta 的一部分。在正式发布之前 API 可能发生变化。Python SDK 会为 `client.beta` 下的每个方法自动加上所需的 `anthropic-beta` 请求头。

## 前置条件

- 一个可访问 Claude Managed Agents beta 的 Anthropic API 密钥。请把它设置为 `ANTHROPIC_API_KEY` 环境变量。
- Python 3.11 或更高版本。
- Anthropic Python SDK。记忆存储方法需要较新的版本：

```bash
uv add anthropic
# or: pip install -U anthropic
```

## 设置客户端

In [ ]:
%%capture
%pip install -q "anthropic>=0.91.0"

In [1]:
import os

from anthropic import Anthropic
from utilities import wait_for_idle_status

MODEL = os.environ.get("COOKBOOK_MODEL", "claude-sonnet-4-6")

client = Anthropic()

### 处理对话轮次的辅助函数

托管的智能体会话是事件驱动的：你发送一条用户消息，然后持续接收流式事件，直到会话进入空闲状态。为了让本指南其余部分更易读，我们把那个循环封装进一个辅助函数，它会打印智能体的回复，以及它在 `/mnt/memory/` 下触碰过的文件。

In [2]:
def run_turn(session_id: str, user_text: str) -> str:
    """Send one user message and stream the agent's response until it goes idle.

    Returns the agent's full text reply for callers who want it programmatically.
    """
    print(f"\n[user]  {user_text}")
    reply_parts: list[str] = []

    with client.beta.sessions.events.stream(session_id) as stream:
        client.beta.sessions.events.send(
            session_id,
            events=[
                {
                    "type": "user.message",
                    "content": [{"type": "text", "text": user_text}],
                }
            ],
        )
        for event in stream:
            if event.type == "agent.message":
                for block in event.content:
                    if block.type == "text":
                        reply_parts.append(block.text)
                        print(f"[agent] {block.text}")

            elif event.type == "agent.tool_use":
                # Surface reads and writes to the memory mount so you can see
                # the agent checking and updating its memory.
                inp = event.input or {}
                target = inp.get("file_path") or inp.get("command", "")
                if "/mnt/memory/" in str(target):
                    print(f"  [memory] {event.name}: {target}")

            elif event.type == "session.status_idle":
                # Break on any idle reason so an unexpected stop_reason
                # (such as requires_action) cannot hang the loop.
                break

            elif event.type == "session.status_terminated":
                break

    wait_for_idle_status(client, session_id)
    return "".join(reply_parts)

## 第 1 步：创建记忆存储

记忆存储是一个按工作空间范围隔离的、用于存放文本文件的具名容器。在生产部署中，你通常会为每个最终用户创建一个存储，并在自己的数据库里维护从你的用户 ID 到存储 ID 的映射。

你在这里设置的 `description` 会在该存储被挂载时渲染进智能体的系统提示词，所以请用它来告诉 Claude 这个存储的用途。

In [3]:
store = client.beta.memory_stores.create(
    name="Shopper Preferences",
    description=(
        "Personal shopping preferences for a single customer: "
        "sizes, style, budget, favorite brands, and materials to avoid."
    ),
)

print(store.id)  # memstore_01...

memstore_01NKkumXZXY8mEhoRA3xhBvN


## 第 2 步：定义购物智能体

每个托管智能体会话都需要一个**智能体**（模型、系统提示词和工具）和一个**环境**（智能体运行所在的容器）。你可以创建一次，并在众多客户与会话之间复用。

给智能体内置的 `agent_toolset`，它包含用于读写记忆的文件工具。`agent_toolset_20260401` 这个类型字符串是该工具集的 API 标识符，不是模型别名，因此即使发布了更新的模型，也不需要更新它。

In [4]:
environment = client.beta.environments.create(
    name="shopping-demo",
    config={"type": "cloud", "networking": {"type": "limited"}},
)

agent = client.beta.agents.create(
    name="Personal Shopper",
    # Tells Anthropic which cookbook this agent came from. Safe to remove.
    metadata={"anthropic_cookbook": "claude-cookbooks/cma-remember-user-preferences"},
    model=MODEL,
    system=(
        "You are a personal shopping assistant for a retail brand. "
        "Help the customer find products that match their taste and budget, "
        "and remember what you learn about them for future visits."
    ),
    tools=[
        {
            "type": "agent_toolset_20260401",
            "default_config": {
                "enabled": True,
                "permission_policy": {"type": "always_allow"},
            },
        }
    ],
)

print(agent.id)  # agent_01...

agent_011CaMbJoHMUyR5TjeFDtMG9


## 第 3 步：首次访问，智能体在此了解偏好

现在启动一个会话，并通过 `resources` 数组挂载记忆存储。`instructions` 字段是针对该次挂载的指引，用来告诉 Claude 在这个会话里如何使用这个特定的存储。

由于这是这位客户首次光临，存储是空的。请留意输出中的 `[memory]` 行：智能体查看了存储，暂时什么也没找到，随后写下一个新文件，记录它学到的内容。

In [5]:
memory_resource = {
    "type": "memory_store",
    "memory_store_id": store.id,
    "access": "read_write",
    "instructions": (
        "This customer's personal preferences: sizes, style, budget, and "
        "materials to avoid. Check it at the start of every conversation "
        "and update it whenever you learn something new."
    ),
}

session_one = client.beta.sessions.create(
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    environment_id=environment.id,
    resources=[memory_resource],
)

# The response tells you where the store is mounted inside the agent's environment.
for resource in session_one.resources:
    if resource.type == "memory_store":
        print(f"Mounted at {resource.mount_path}")

Mounted at /mnt/memory/shopper-preferences


In [6]:
run_turn(
    session_one.id,
    "Hi! I'm looking for a new jacket. A few things about me: I wear a size "
    "medium, I only buy vegan leather (no animal leather please), my budget "
    "is usually under $200, and I love earth tones. What would you suggest?",
)

[user]  Hi! I'm looking for a new jacket. A few things about me: I wear a size medium, I only buy vegan leather (no animal leather please), my budget is usually under $200, and I love earth tones. What would you suggest?
[agent] Let me check if I have any info about you on file, and save your preferences right away!
  [memory] bash: cat /mnt/memory/shopper-preferences 2>/dev/null || echo "No file found"
  [memory] write: /mnt/memory/shopper-preferences
  [memory] write: /mnt/memory/shopper-preferences/preferences.md
[agent] I've saved your preferences for future visits! Now, here are some great **vegan leather jacket picks in earth tones under $200** that I'd suggest:

---

### 🌿 Top Picks for You

#### 1. **Olive Moto Jacket** — ~$110–$150
A classic biker silhouette in a deep **olive green** faux leather. Look for styles with zip-detail sleeves and a fitted waist. Great for casual or edgy looks.
- *Why you'll love it:* Earthy tone, very on-trend, versatile for layering.

#### 2. **Cam

## 第 4 步：查看智能体保存了什么

智能体写入该挂载点的一切都是普通的记忆文档，你的应用可以通过 API 读取、编辑或删除它们。用 `view="full"` 列出存储内容，以便在响应中包含文件内容。

这就是你在产品中构建“我们对你了解的内容”页面、把记忆同步到你自己的数据库，或让人工审核者纠正智能体写错内容的方式。

In [7]:
page = client.beta.memory_stores.memories.list(
    store.id,
    view="full",
)

for memory in page.data:
    if memory.type == "memory":
        print(f"=== {memory.path} ===")
        print(memory.content)
        print()

=== /preferences.md ===
# Shopper Preferences

## Sizes
- Tops/Jackets: Medium

## Style
- Loves earth tones (browns, tans, olive, terracotta, camel, rust, etc.)
- Interested in jackets

## Budget
- Usually under $200

## Materials
- VEGAN LEATHER ONLY — absolutely no animal leather
- (No other material restrictions noted yet)

## Favorite Brands
- None noted yet

## Other Notes
- First visit; preferences collected 2026-04-23


你会看到一个文件（通常是类似 `/preferences.md` 的名字），其中按主题组织了客户提到的尺码、预算、材质和颜色偏好。文件名和结构都是 Claude 自行选择的。

## 第 5 步：再次访问，智能体自行调取记忆

这是最关键的时刻。创建一个**全新的会话**，并挂载**同一个记忆存储**。客户没有重复任何偏好，但智能体从记忆中读取了它们，并据此定制它的推荐。

In [8]:
session_two = client.beta.sessions.create(
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    environment_id=environment.id,
    resources=[memory_resource],  # same store, new session
)

run_turn(
    session_two.id,
    "Hey, I'm back! I need a bag for work. Any recommendations?",
)

[user]  Hey, I'm back! I need a bag for work. Any recommendations?
[agent] Welcome back! Let me check your preferences before making any recommendations!
  [memory] read: /mnt/memory/shopper-preferences
  [memory] bash: ls /mnt/memory/shopper-preferences/
  [memory] read: /mnt/memory/shopper-preferences/preferences.md
[agent] Great news — I already have your preferences on file! Here's what I kept in mind for you:

- 🎨 **Style:** Earth tones (browns, tans, olive, camel, etc.)
- 💰 **Budget:** Under $200
- 🌱 **Materials:** Vegan leather only — no animal leather

With that in mind, here are some work bag recommendations that would suit you perfectly:

---

### 🛍️ Top Picks for a Work Bag

1. **Structured Tote in Camel Vegan Leather** — A polished, professional look in a warm earth tone. Fits a laptop + daily essentials. Great desk-to-commute bag.

2. **Olive Canvas & Vegan Leather Trim Laptop Bag** — A sleek crossbody/shoulder bag with a relaxed but put-together vibe. The canvas keeps it 

在输出中请注意：智能体在回答前先读取了 `/mnt/memory/shopper-preferences/`，随后推荐了纯素皮革、大地色系且低于 200 美元的包，尽管这条消息完全没有提到这些因素。偏好是从第一个会话延续过来的。

## 更进一步：面向生产的模式

### 用你已有的数据预置存储

如果你已经从客户账户资料或购买历史中了解了一些信息，可以在第一个会话开始之前把它们写入存储，让智能体一开始就掌握情况。在真实应用中，你会在任何会话创建之前执行这个预置步骤；这里把它放在演示之后，只是为了让上面“先学习、再调取”的主线流程保持聚焦。

> **说明：** 请在 Notebook 末尾的清理单元之前运行此单元，因为 `store` 会在清理单元中被删除。

In [9]:
# Optional: run before the cleanup cell.
seeded = client.beta.memory_stores.memories.create(
    store.id,
    path="/purchase-history.md",
    content=(
        "## Recent purchases\n"
        "- Canvas tote, olive, $89 (Jan 2026)\n"
        "- Wool beanie, rust, $34 (Dec 2025)\n"
    ),
)
print(f"Seeded {seeded.path}")

Seeded /purchase-history.md


### 组合按客户存储与共享存储

一个会话最多可以挂载八个记忆存储，每个都有自己的访问级别。一种常见模式是：每位客户一个读写存储，再加一个存放品牌级知识的只读存储，供所有会话共享。

```python
catalog = client.beta.memory_stores.create(
    name="Product Catalog Notes",
    description="Current promotions, sizing guidance, and stock notes.",
)

session = client.beta.sessions.create(
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    environment_id=environment.id,
    resources=[
        {
            "type": "memory_store",
            "memory_store_id": store.id,
            "access": "read_write",
            "instructions": "This customer's personal preferences.",
        },
        {
            "type": "memory_store",
            "memory_store_id": catalog.id,
            "access": "read_only",
            "instructions": "Brand-wide product guidance. Consult before recommending items.",
        },
    ],
)

# Remember to clean up the catalog store when you are done:
# client.beta.memory_stores.delete(catalog.id)
```

### 审计与纠正

对记忆存储的每次写入都会记录为一个不可变版本，并附带产生它的会话。用 `client.beta.memory_stores.memory_versions.list(...)` 查看历史，用 `client.beta.memory_stores.memories.update(...)` 纠正智能体写错的文件。完整接口请参见 [Memory API 参考](https://docs.anthropic.com/en/docs/managed-agents/memory)。

## 清理

删除你在跟随本指南操作时创建的资源。

In [10]:
wait_for_idle_status(client, session_one.id)
wait_for_idle_status(client, session_two.id)

client.beta.sessions.archive(session_one.id)
client.beta.sessions.archive(session_two.id)
client.beta.memory_stores.delete(store.id)
client.beta.agents.archive(agent.id)
client.beta.environments.archive(environment.id)

## 总结

你构建了一个能在多次访问之间记住客户的购物助手，方法是：

1. 为该客户创建记忆存储
2. 通过 `resources` 把它挂载到每个会话
3. 让 Claude 在学到新偏好时读写 `/mnt/memory/` 下的文件
4. 通过 Memory API 从你自己的应用中查看这些文件

在此基础上，你可以把自己的用户 ID 映射到记忆存储 ID，用已有的客户数据预置存储，并在其上叠加共享的只读存储来承载品牌级知识。

### 本系列的其他 Notebook

- [`CMA_iterate_fix_failing_tests.ipynb`](CMA_iterate_fix_failing_tests.ipynb) — 入门 Notebook。通过在一个失败测试套件上进行“执行 → 观察 → 修复”循环，介绍智能体、环境、会话、文件挂载以及流式事件循环。
- [`CMA_operate_in_production.ipynb`](CMA_operate_in_production.ipynb) — 生产环境配置：由密钥库支撑的 MCP 凭据、用于在不保持长连接情况下实现人工介入（HITL）的 `session.status_idled` webhook，以及资源生命周期的 CRUD 动词。

### 延伸阅读

- [Claude Managed Agents 概览](https://docs.anthropic.com/en/docs/managed-agents/overview)
- [记忆存储 API 参考](https://docs.anthropic.com/en/docs/managed-agents/memory)
- [会话资源](https://docs.anthropic.com/en/docs/managed-agents/memory#attach-a-memory-store-to-a-session)